# Open Source, Local, Token Cost

In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

## Connect to OpenRouter

In [4]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = os.getenv('https://openrouter.ai/api/v1')

if openrouter_api_key:
    print('OpenRouter API key loaded successfully,')
else:
    print('Failed to load OpenRouter API key')

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, 
                    api_key=openrouter_api_key,
                    timeout=120,
                    max_retries=5)

OpenRouter API key loaded successfully,


## Part 1: The Data And A Smaller Prompt

In [6]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False,
                    held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id. 
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """

    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [7]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim['claim_id']]['difficulty'] != 'manual']
policies = {policy['policy_id']: policy for policy in load('policies.json')}
print(f"{len(claims)} claims")

20 claims


In [9]:
decisions = pd.DataFrame([labels[claim['claim_id']] for claim in claims])
decisions['decision'].value_counts()

decision
approve    12
decline     5
refer       3
Name: count, dtype: int64

### A Smaller Prompt

In [10]:
policy_wording= (DATA / 'policy_wording.md').read_text(encoding='utf-8')
short_wording = (DATA / 'policy_wording_short.md').read_text(encoding='utf-8')

print(f'The policy wording is {len(policy_wording.split()):,} words')
print(f'The short policy wording is {len(short_wording.split()):,} words')

The policy wording is 1,369 words
The short policy wording is 808 words


In [11]:
def split_sections(wording):
    """ Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "##" + part for part in parts[1:]}
    return parts[0], sections

opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [12]:
BENEFITS = {
    'trip_cancellation': 'Cancellation and Interruption',
    'trip_interruption': 'Cancellation and Interruption',
    'trip_delay': 'Trip Delay',
    'baggage_delay': 'Baggage Delay',
    'baggage': 'Baggage',
    'passport': 'Passport',
    'emergency_medical': 'Emergency Medical' 
}

def make_wording(claim):
    """ The sections of the short wording that this claim needs: the limits, 
    its benefits, the add-ons if the policy has any, and the rules for  every claim.
    """
    headings = ['Limits', BENEFITS[claim['claim_type']]]
    if policies[claim['policy_id']]['add_ons']:
        headings.append('Add-ons')
    headings += ['Never covered', 'Payment', 'Claims']
    return '\n\n'.join([opening.strip()] + [sections[heading].strip()for heading in headings])

In [13]:
INSTRUCTIONS = """ You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.
"""

def make_user_prompt(claim):
    """ The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim['policy_id']]
    record = {key: value for key, value in policy.items() if key not in ['policy_id', 'customer_id', 'sold_through', 'status']}
    details = {key: value for key, value in claim.items() if key not in ['policy_id', 'customer_id', 'status']}
    details['documents'] = [{'name': document['name'], 'type': document['type']} for document in claim['documents']]
    return f""" # Policy record
    
{json.dumps(record)}

# Claim

{json.dumps(details)}
"""

def make_messages(claim):
    """ Build the messages for one claim: the task and the the wording it needs,
    then the policy record and the claim.
    """
    return [
        {
            'role': 'system',
            'content': f"{INSTRUCTIONS}\n{make_wording(claim)}\n"
        },
        {
            'role': 'user',
            'content': make_user_prompt(claim)
        },
    ]

In [14]:
messages = make_messages(claims[0])
print(messages[0]['content'])
print(messages[1]['content'])

 You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

##Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Baggage Delay ma

## Part 2: Six Open-Weight Models on OpenRouter